# Zirconium loop line broadening derivation

This notebook derives a parameterized XRD line-broadening workflow for zirconium containing loop-driven anisotropic strain broadening.


## Section 1 — Thesis map and study plan

This section now maps the notebook workflow to thesis source blocks and defines the minimum study plan before any zirconium-specific inversion is attempted.

### 1.1 Thesis-to-workflow map (explicit pointers)

| Notebook task | Thesis anchor to check first | Why it matters for this notebook |
|---|---|---|
| Line-profile/Fourier formalism | Thesis source blocks **[S4], [S5], [S6], [S8]** | Defines reciprocal-space profile description and Fourier treatment of size/strain broadening. |
| Dislocation strain and Wilkens modeling | **[S4], [S6], [S14]** | Provides the Krivoglaz–Wilkens style basis and arrangement-parameter logic used in strain broadening. |
| Whole-profile fitting strategy | **MWP: [S4], [S6]**; **CMWP: [S14]** | Connects equation-level broadening terms to full-pattern fitting logic. |
| Dislocation-type sensitivity in HCP systems | **[S11], [S12], [S22]** | Motivates separation of Burgers-vector-family-dependent contributions (relevant to a/c discrimination). |

> **Usage note:** these thesis anchors must be cross-checked directly in the thesis PDF/HTML when finalizing any publishable derivation.

### 1.2 Physical-role summary used in this notebook

- **Line-profile coordinates:** connect measured intensity in 2θ-space to reciprocal-space variables used by Fourier/Warren–Averbach style treatments.
- **Size broadening:** finite coherent domain size contributes a reflection-dependent peak-width term.
- **Strain broadening:** heterogeneous strain fields (including dislocation fields) broaden diffraction peaks.
- **Dislocation density (ρ):** scales strain amplitude in dislocation broadening models.
- **Contrast factors (C or C_hkl):** encode reflection-dependent coupling between dislocation strain fields and scattering vector orientation.
- **Hexagonal anisotropy:** in zirconium alloys, reflection dependence is not isotropic; orientation terms must be retained.
- **Individual vs average contrast factors:** strongly textured specimens require reflection-wise **C_hkl** treatment instead of only averaged C̄.
- **MWP/CMWP logic:** links physically parameterized broadenings to whole-profile/pattern simulations and fitting.

### 1.3 Concrete study plan for a-type vs c-type loops in textured zirconium

1. Assemble thesis equations and verify symbol conventions (q, β, Fourier coefficients, Wilkens parameters).
2. Define required external inputs (elastic/contrast constants, Burgers vectors, texture descriptors, instrument model).
3. Compute reflection-wise C_hkl for selected zirconium reflections under texture assumptions.
4. Build separate a-type and c-type strain-broadening channels using the same base formalism.
5. Form the mixed-population broadening model and derive the symbolic a:c inversion expression.
6. Run parameterized simulations first; perform numeric a:c estimation only after zirconium-specific inputs are validated.

### 1.4 Assumption boundary

We model observed broadening as a combination of instrumental, size, and strain terms:

\[
eta_{\mathrm{obs}}^2 \approx eta_{\mathrm{inst}}^2 + eta_{\mathrm{size}}^2 + eta_{\mathrm{strain}}^2
\]

All zirconium-specific constants remain external inputs (not hard-coded).



In [ ]:
import numpy as np

beta_inst = 0.0018
beta_size = 0.0021
beta_strain = 0.0014
beta_obs = np.sqrt(beta_inst**2 + beta_size**2 + beta_strain**2)
beta_obs


## Section 2 — Required external inputs

| Input | Symbol | Meaning | Units | Source |
|---|---|---|---|---|
| X-ray wavelength | $\lambda$ | Probe wavelength | m | Instrument setup |
| Zirconium lattice parameters | $a, c$ | HCP cell parameters | m | Refined or literature |
| Contrast baseline | $\bar{C}$ | Average dislocation contrast | – | Elastic/dislocation model |
| Contrast anisotropy coefficient | $q$ | Reflection anisotropy term | – | Elastic/dislocation model |
| Burgers vector magnitude | $b$ | Loop/dislocation Burgers magnitude | m | Defect model |
| Dislocation/loop density | $\rho$ | Defect density | m$^{-2}$ | Fitted or independent estimate |
| Shape factor | $K$ | Scherrer constant | – | Profile model assumption |
| Instrument broadening coefficients | $U,V,W$ | Caglioti parameters | rad$^2$ | Instrument calibration |


## Section 3 — Reciprocal-space transform

\[
q = \frac{4\pi}{\lambda}\sin\theta,\qquad 2\theta = 2\arcsin\left(\frac{q\lambda}{4\pi}\right)
\]


In [ ]:
from src.xrd_line_broadening import two_theta_to_q, q_to_two_theta

wavelength = 1.5406e-10

two_theta = np.deg2rad(np.array([30.0, 40.0, 50.0]))
q = two_theta_to_q(two_theta, wavelength)
two_theta_back = q_to_two_theta(q, wavelength)
q, np.rad2deg(two_theta_back)


## Section 4 — Size broadening (Scherrer)

\[
\beta_{\mathrm{size}} = \frac{K\lambda}{L\cos\theta}
\]


In [ ]:
from src.xrd_line_broadening import scherrer_fwhm

K = 0.9
L = 80e-9
theta = np.deg2rad(np.array([15.0, 20.0, 25.0]))
beta_size = scherrer_fwhm(theta, wavelength=wavelength, domain_size=L, k=K)
beta_size


## Section 5 — Strain broadening and contrast factors

This section addresses **Recommendation 2** from `prompt_compliance_review.md`: add an explicit Warren–Averbach to Krivoglaz–Wilkens derivation chain with executable equations.

### 5.1 Warren–Averbach factorization

For a reflection with reciprocal-lattice magnitude $g$, the normalized Fourier coefficient is factored as

\[
A(L,g)=A_S(L)\,A_D(L,g),
\]

where:
- $A_S(L)$ is the size Fourier coefficient,
- $A_D(L,g)$ is the strain Fourier coefficient,
- $L$ is Fourier length.

Using the common exponential size approximation:

\[
A_S(L)=\exp\!\left(-\frac{L}{D_A}\right),
\]

and the strain coefficient form:

\[
A_D(L,g)=\exp\!\left[-2\pi^2 g^2\langle\varepsilon^2(L)\rangle\right].
\]

Therefore:

\[
A(L,g)=\exp\!\left(-\frac{L}{D_A}\right)
\exp\!\left[-2\pi^2 g^2\langle\varepsilon^2(L)\rangle\right].
\]

Taking logarithms (non-trivial algebra step retained):

\[
\ln A(L,g)=-\frac{L}{D_A}-2\pi^2 g^2\langle\varepsilon^2(L)\rangle.
\]

At fixed $L$, plotting $\ln A$ versus $g^2$ gives a slope:

\[
\frac{\partial \ln A}{\partial g^2}=-2\pi^2\langle\varepsilon^2(L)\rangle,
\]

which isolates strain from size broadening.

### 5.2 Krivoglaz–Wilkens link for dislocation strain

For dislocation-controlled strain broadening in anisotropic crystals, the mean square strain scales with
$\rho b^2 C_{hkl}$ and an arrangement/correlation kernel.

A transparent asymptotic Wilkens-style representation used here is:

\[
\langle\varepsilon^2(L)\rangle
\approx
\frac{\rho b^2 C_{hkl}}{4\pi}
L^2\ln\!\left(\frac{R_e}{L}\right),
\]

where:
- $\rho$ = dislocation density,
- $b$ = Burgers magnitude,
- $C_{hkl}$ = reflection-specific contrast factor,
- $R_e$ = effective outer cut-off / arrangement length.

Substitute into the WA logarithmic equation:

\[
\ln A(L,g)=
-\frac{L}{D_A}
-2\pi^2 g^2
\left[
\frac{\rho b^2 C_{hkl}}{4\pi}
L^2\ln\!\left(\frac{R_e}{L}\right)
\right].
\]

Collecting constants:

\[
\ln A(L,g)=
-\frac{L}{D_A}
-
\frac{\pi}{2}
\,g^2\rho b^2 C_{hkl}
L^2\ln\!\left(\frac{R_e}{L}\right).
\]

This equation is the operational bridge from measured Fourier coefficients to
$(\rho, b, C_{hkl}, R_e)$ under the adopted approximation.

### 5.3 Reflection anisotropy (hexagonal form)

\[
C_{hkl}=\bar{C}\left(1-q\,\eta_{hkl}\right),\qquad
\eta_{hkl}=\frac{\frac{4}{3}(h^2+hk+k^2)}{\frac{4}{3}(h^2+hk+k^2)+(lc/a)^2}.
\]

In textured zirconium alloys, reflection-wise $C_{hkl}$ must be kept explicit rather than replaced by a single average $\bar{C}$.


### 5.4 Hexagonal texture workflow (reflection-wise $C_{hkl}$)

For strongly textured zirconium alloys, use a **variant-resolved** workflow:

1. Choose measured reflections $\{hkl\}$.
2. For each texture variant $v$, compute $C^{(v)}_{hkl}$ (not only a global average).
3. Compute effective reflection-wise contrasts using ODF/volume weights $w_v$:

\[
C^{\mathrm{eff}}_{hkl}=\sum_v w_v C^{(v)}_{hkl}, \qquad \sum_v w_v=1.
\]

4. Insert $C^{\mathrm{eff}}_{hkl}$ into the WA→Wilkens strain term for each reflection.

This is the practical bridge from thesis anisotropy equations to textured-specimen analysis,
and it is where orientation relationships between loop families manifest as reflection-dependent
broadening contrasts.


### 5.5 Equation-to-code trace tags (Recommendation 4)

To improve thesis traceability, equation IDs are linked directly to implementation calls.

| Eq. tag (this notebook) | Equation | Code implementation |
|---|---|---|
| **Eq. (E5.1)** | $A(L,g)=A_S(L)A_D(L,g)$ | `wa_total_fourier_coefficient(...)` |
| **Eq. (E5.2)** | $A_S(L)=\exp(-L/D_A)$ | `wa_size_fourier_coefficient(...)` |
| **Eq. (E5.3)** | $A_D(L,g)=\exp[-2\pi^2 g^2\langle\varepsilon^2(L)\rangle]$ | `wa_strain_fourier_coefficient(...)` |
| **Eq. (E5.4)** | $\langle\varepsilon^2(L)\rangle\approx(\rho b^2 C_{hkl}/4\pi)L^2\ln(R_e/L)$ | `wilkens_mean_square_strain_asymptotic(...)` |
| **Eq. (E5.5)** | $C_{hkl}=\bar{C}(1-q\eta_{hkl})$ | `contrast_factor_hex(...)` |
| **Eq. (E5.6)** | $C^{\mathrm{eff}}_{hkl}=\sum_v w_v C^{(v)}_{hkl}$ | `texture_weighted_contrast(...)` |

The code cell below includes inline comments with these tags to keep derivation-to-code mapping explicit.


In [ ]:
from src.xrd_line_broadening import (
    microstrain_fwhm,
    wa_size_fourier_coefficient,
    wa_strain_fourier_coefficient,
    wa_total_fourier_coefficient,
    wilkens_mean_square_strain_asymptotic,
)
from src.contrast_factors import HKL, contrast_factor_hex, texture_weighted_contrast

# Example reflection and external-input placeholders (replace with validated zirconium inputs)
hkl = HKL(1, 0, 1)
c_over_a = 1.593
c_bar = 0.28
q_aniso = 1.6
# Eq. (E5.5)
C_hkl = contrast_factor_hex(hkl.h, hkl.k, hkl.l, c_over_a=c_over_a, c_bar=c_bar, q=q_aniso)

# Eq. (E5.1-E5.3): Warren-Averbach variables
L = np.linspace(1e-9, 120e-9, 200)  # Fourier lengths [m]
D_A = 80e-9                          # area-weighted domain size [m]
g = (2.0 * np.sin(theta)) / wavelength  # reciprocal lattice magnitude [1/m]

# Eq. (E5.4): Krivoglaz-Wilkens asymptotic mean-square strain (parameterized)
rho = 1.0e14      # [m^-2], placeholder
b = 3.2e-10       # [m], placeholder
R_e = 300e-9      # [m], placeholder

# Compute one <eps^2(L)> curve per reflection sample for demonstration
eps2 = wilkens_mean_square_strain_asymptotic(
    fourier_length=L,
    rho=rho,
    burgers=b,
    contrast_factor=C_hkl,
    outer_cutoff_radius=R_e,
)

A_S = wa_size_fourier_coefficient(L, area_weighted_domain_size=D_A)
A_D = wa_strain_fourier_coefficient(reciprocal_lattice=g[0], mean_square_strain=eps2)
A_total = wa_total_fourier_coefficient(
    fourier_length=L,
    reciprocal_lattice=g[0],
    area_weighted_domain_size=D_A,
    mean_square_strain=eps2,
)

# Optional isotropic line-width reference retained for continuity with prior cells
epsilon = 8e-4
beta_strain = microstrain_fwhm(theta, microstrain_rms=epsilon)

A_S[:3], A_D[:3], A_total[:3], beta_strain

# --- Strong-texture example: reflection-wise C_hkl via variant weighting ---
hkls = [HKL(1, 0, 0), HKL(0, 0, 2), HKL(1, 0, 1)]

# Example texture variants (parameterized sensitivity placeholders)
variant_params = [
    {"c_bar": 0.30, "q": 1.40},
    {"c_bar": 0.26, "q": 1.75},
    {"c_bar": 0.28, "q": 1.55},
]
variant_weights = np.array([0.55, 0.30, 0.15])

contrast_by_variant = np.array([
    [contrast_factor_hex(h.h, h.k, h.l, c_over_a=c_over_a, c_bar=v["c_bar"], q=v["q"]) for h in hkls]
    for v in variant_params
])

# Eq. (E5.6)
C_eff_hkl = texture_weighted_contrast(contrast_by_variant, variant_weights, axis=0)

contrast_by_variant, C_eff_hkl


## Section 6 — Ratio framework for mixed a/c loop broadening

Starting from thesis-style strain-broadening scaling for each loop family $i\in\{a,c\}$:

\[
\beta_i^2(hkl) \propto b_i^2\,\rho_i\,C^{(i)}_{hkl}\,\Phi_i,
\]

where $b_i$ is Burgers-vector magnitude, $\rho_i$ is effective dislocation density,
$C^{(i)}_{hkl}$ is reflection-specific contrast, and $\Phi_i$ collects arrangement/correlation
terms (e.g., Wilkens-like arrangement parameter effects).

Define three ratios explicitly:

1. **Ratio of broadening contributions**
   \[
   R_\beta(hkl) \equiv \frac{\beta_a^2(hkl)}{\beta_c^2(hkl)}
   = \frac{b_a^2\rho_a C^{(a)}_{hkl}\Phi_a}{b_c^2\rho_c C^{(c)}_{hkl}\Phi_c}.
   \]

2. **Ratio of effective dislocation densities**
   \[
   R_\rho \equiv \frac{\rho_a}{\rho_c}.
   \]

3. **Ratio of loop populations**
   \[
   R_N \equiv \frac{N_a}{N_c},
   \]
   with mapping to density ratio requiring additional geometric assumptions.

Symbolic superposition for mixed a/c contributions:

\[
\beta_{\mathrm{strain,mix}}^2(hkl)=\beta_a^2(hkl)+\beta_c^2(hkl)
=K\left[b_a^2\rho_a C^{(a)}_{hkl}\Phi_a+b_c^2\rho_c C^{(c)}_{hkl}\Phi_c\right],
\]

or in ratio form using $R_\rho$:

\[
\beta_{\mathrm{strain,mix}}^2(hkl)=K\rho_c\left[b_a^2R_\rho C^{(a)}_{hkl}\Phi_a+b_c^2C^{(c)}_{hkl}\Phi_c\right].
\]

### Conditions for equivalence of ratios

To map one ratio into another, the following assumptions are required:

- **$R_\beta \leftrightarrow R_\rho$** requires known (or equal) $b_a,b_c$; known
  reflection-wise $C^{(a)}_{hkl},C^{(c)}_{hkl}$; and known/equal arrangement terms
  $\Phi_a,\Phi_c$ for the compared state/reflection.
- **$R_\rho \leftrightarrow R_N$** requires a constitutive link between loop count and
  effective density (habit-plane fractions, loop-size distributions, line length per loop,
  and visibility/effectiveness factors).
- **$R_\beta \leftrightarrow R_N$** requires both assumption sets above simultaneously.

### Missing zirconium-specific inputs for unique numeric $a$ estimate

| Missing zirconium-specific input | Role in model | Required for unique numeric $a$ estimate |
|---|---|---|
| Loop habit statistics (family fractions on basal/prismatic/pyramidal variants) | Converts population ratios to effective density terms | **Required** |
| Reflection-specific contrast factors per loop family, $C^{(a)}_{hkl},C^{(c)}_{hkl}$ | Couples anisotropy to each measured reflection | **Required** |
| Arrangement/correlation parameters per family ($\Phi_a,\Phi_c$ or equivalent Wilkens terms) | Controls screening/range effects in strain broadening | **Required** |
| Burgers-vector magnitudes and character fractions for each active family | Sets prefactor $b_i^2$ and mixed-character weighting | **Required** |
| Texture/ODF coupling assumptions for reflection weighting | Maps single-crystal contrast to polycrystal measured intensity/broadening | **Required** |
| Loop size distribution statistics by family | Needed to convert loop counts into line length/density and check consistency with size broadening | **Required** |
| Instrument-function deconvolution uncertainty bounds | Needed to isolate strain term without bias in inferred $a$ | **Required** |

> Downstream note: the Caglioti instrumental model is retained in the next code cell for
> execution of later composite-width examples.


In [ ]:
from src.xrd_line_broadening import caglioti_fwhm

U, V, W = 1.0e-5, 2.0e-5, 4.0e-5
beta_inst = caglioti_fwhm(theta, U, V, W)
beta_inst


## Section 7 — Composite width equations

For Gaussian-like components:
\[
\beta_{\mathrm{obs}} = \sqrt{\beta_{\mathrm{inst}}^2 + \beta_{\mathrm{size}}^2 + \beta_{\mathrm{strain}}^2}
\]

For Lorentzian-like components:
\[
\beta_{\mathrm{obs}} = \beta_{\mathrm{inst}} + \beta_{\mathrm{size}} + \beta_{\mathrm{strain}}
\]


In [ ]:
from src.xrd_line_broadening import combine_fwhm_gaussian, combine_fwhm_lorentzian

beta_obs_g = combine_fwhm_gaussian(beta_inst, beta_size, beta_strain)
beta_obs_l = combine_fwhm_lorentzian(beta_inst, beta_size, beta_strain)
beta_obs_g, beta_obs_l


## Section 8 — Two-population a/c loop contribution model

### Simulation label: **Parameterized sensitivity study**

The numerical example below uses illustrative scaling factors (e.g., 1.10, 0.90, 1.25) and
assumed weights. Therefore it is a **parameterized sensitivity study**, not a fully
thesis-supported prediction.

For completeness, the mixed-width form remains:

\[
\beta_{\mathrm{mix}} = \frac{w_a}{w_a+w_c}\beta_a + \frac{w_c}{w_a+w_c}\beta_c
\]

A simulation may be labeled **fully thesis-supported** only when all required zirconium
inputs listed in Section 6 are supplied from validated sources.


In [ ]:
from src.diffraction_simulation import two_population_ac_broadening

# Parameterized sensitivity study (illustrative factors; not uniquely identified).
size_a = beta_size * 1.10
strain_a = beta_strain * 0.90
size_c = beta_size * 0.95
strain_c = beta_strain * 1.25

beta_mix = two_population_ac_broadening(
    theta_rad=theta,
    instrument_fwhm_rad=beta_inst,
    size_a=size_a,
    strain_a=strain_a,
    size_c=size_c,
    strain_c=strain_c,
    weight_a=0.65,
    weight_c=0.35,
)

beta_mix


## Section 9 — Synthetic profile construction and execution checklist

Peak profile (pseudo-Voigt):
\[
I(2\theta)=\eta I_L(2\theta)+(1-\eta)I_G(2\theta)
\]

Practical execution order:
1. Enter all required external zirconium inputs.
2. Compute reflection-wise $C_{hkl}$.
3. Build size and strain widths for each reflection.
4. Combine with instrumental broadening.
5. Simulate/fitted synthetic peaks for interpretation.


In [ ]:
from src.diffraction_simulation import build_population_peaks, synthetic_pattern

x = np.deg2rad(np.linspace(25, 65, 2000))
centers = np.deg2rad(np.array([30.0, 36.0, 48.0, 57.5]))
amplitudes = np.array([1200.0, 900.0, 1400.0, 850.0])
widths = np.interp(centers, np.linspace(theta.min(), theta.max(), len(beta_mix)), beta_mix)
peaks = build_population_peaks(centers, amplitudes, widths)
pattern = synthetic_pattern(x, peaks, background=45.0)

pattern[:5], pattern.max()
